# 🌿 03 Testing — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/03_testing.ipynb`                 |
| **Tujuan** | Muat model, jalankan prediksi pada data testing, simpan hasil. |
| **Input**  | `artifacts/model_bonsai_lstm.keras`, `artifacts/data_test.npz`, `artifacts/scaler_bonsai.pkl`, `artifacts/label_info.json` |
| **Output** | `artifacts/predictions.csv`, `artifacts/y_test_labels.csv` |
| **Urutan** | Jalankan setelah: `02_training.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import random
import warnings
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR = "../artifacts"
SOIL_MIN      = 0.0
SOIL_MAX      = 100.0

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("✅ Konfigurasi testing siap.")

In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
import json, joblib

REQUIRED = [
    f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras",
    f"{ARTIFACTS_DIR}/data_test.npz",
    f"{ARTIFACTS_DIR}/scaler_bonsai.pkl",
    f"{ARTIFACTS_DIR}/label_info.json",
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, f"⛔ Artefak tidak ada: {missing}"
print("✅ Semua artefak tersedia.")

In [4]:
# ── RULE-TEST-02: Muat Semua Artefak ───────────────────────────────────
test_data  = np.load(f"{ARTIFACTS_DIR}/data_test.npz")
X_test     = test_data["X_test"]
y_test_cls = test_data["y_test_cls"]
y_test_reg = test_data["y_test_reg"]

model  = tf.keras.models.load_model(f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras")
scaler = joblib.load(f"{ARTIFACTS_DIR}/scaler_bonsai.pkl")

with open(f"{ARTIFACTS_DIR}/label_info.json") as f:
    label_info = json.load(f)

print(f"[LOAD] X_test shape : {X_test.shape}")
print(f"[LOAD] Model        : {model.name}")

In [5]:
# ── RULE-TEST-03: Menjalankan Prediksi ────────────────────────────────
y_pred_prob  = model.predict(X_test, verbose=0).flatten()

y_pred_class = (y_pred_prob >= 0.5).astype(int)

y_pred_soil  = y_pred_prob * (SOIL_MAX - SOIL_MIN) + SOIL_MIN

print(f"[PREDICT] Total prediksi       : {len(y_pred_prob)}")
print(f"[PREDICT] Prediksi Siram (1)   : {y_pred_class.sum()}")
print(f"[PREDICT] Prediksi Tidak (0)   : {(y_pred_class==0).sum()}")
print(f"[PREDICT] Probabilitas rata2   : {y_pred_prob.mean():.4f}")

In [6]:
# ── RULE-TEST-04: Simpan Hasil Prediksi ────────────────────────────────
pred_df = pd.DataFrame({
    "y_pred_prob"    : y_pred_prob,
    "y_pred_class"   : y_pred_class,
    "y_pred_soil_pct": y_pred_soil,
})
pred_df.to_csv(f"{ARTIFACTS_DIR}/predictions.csv", index=False)

actual_df = pd.DataFrame({
    "y_true_class"   : y_test_cls,
    "y_true_soil_pct": y_test_reg,
})
actual_df.to_csv(f"{ARTIFACTS_DIR}/y_test_labels.csv", index=False)

print("[SAVE] ✅ artifacts/predictions.csv")
print("[SAVE] ✅ artifacts/y_test_labels.csv")

In [7]:
# ── RULE-TEST-05: Sanity Check Hasil Prediksi ───────────────────────────
check_df = pd.concat([actual_df, pred_df], axis=1)
print("\n[SANITY CHECK] 10 sampel pertama:")
print(check_df.head(10).to_string(index=False))

correct = (check_df["y_true_class"] == check_df["y_pred_class"]).sum()
total   = len(check_df)
print(f"\n[SANITY] Prediksi benar: {correct}/{total} = {correct/total*100:.2f}%")

## 📊 Ringkasan Testing

**File yang dihasilkan:**
- `predictions.csv` → Probabilitas, kelas prediksi, estimasi soil %
- `y_test_labels.csv` → Label aktual & nilai aktual soil %

**Langkah selanjutnya:**
Jalankan `notebooks/04_evaluasi.ipynb` untuk menghitung metrik dan visualisasi.